#Task: File Ingestion and Schema validation
Take any csv/text file of 2+ GB of your choice. --- (You can do this assignment on Google colab)

Read the file ( Present approach of reading the file )

Try different methods of file reading eg: Dask, Modin, Ray, pandas and present your findings in term of computational efficiency

Perform basic validation on data columns : eg: remove special character , white spaces from the col name

As you already know the schema hence create a YAML file and write the column name in YAML file. --define separator of
read and write file, column name in YAML

Validate number of columns and column name of ingested file with YAML.

Write the file in pipe separated text file (|) in gz format.

Create a summary of the file:

Total number of rows,

total number of columns

file size

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
import time

In [5]:
file_path = '/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022.csv'
file_size = os.path.getsize(file_path) / (1024**3)  # size in GB

print(f"File size: {file_size:.2f} GB")

File size: 2.27 GB


## Read in the data with Dask

In [6]:
from dask import dataframe as dd

start = time.time()
dask_df = dd.read_csv(file_path, assume_missing=True)
end = time.time()

print("Read CSV with Dask:", round(end - start, 2), "seconds")

Read CSV with Dask: 0.12 seconds


## Read in the data with Pandas

In [7]:
import pandas as pd

# Read just 100,000 rows for benchmarking
start = time.time()
df_pandas_sample = pd.read_csv(file_path, nrows=1000000)
end = time.time()

print("Read 1000k rows with Pandas:", round(end - start, 2), "seconds")


Read 1000k rows with Pandas: 9.93 seconds


 ### Due to RAM limitations, only 1,000,000 of ~13.6 million rows were read using `pandas` in ~6.87 seconds; estimating linearly, a full read would take ~94 seconds if enough memory were available.

## Read in the data with Modin and Ray

In [8]:
!pip install modin[ray] -q


In [ ]:
import modin.pandas as mpd
import time

start = time.time()
df_modin_sample = mpd.read_csv(file_path, nrows=1000000)  # Read 1 million rows
end = time.time()

print("Read 1M rows with Modin + Ray:", round(end - start, 2), "seconds")



⚠️ Modin with Ray caused runtime crashes in Google Colab, even when attempting to load only a portion of the file.  
> This appears to be a compatibility issue between Modin’s parallelism and Colab’s limited memory and distributed system support.  
> Therefore, we excluded Modin results from this comparison.

# Here Dask is better than Pandas, Modin and Ray, with the least reading time of 0.04 sec

In [9]:
import dask.dataframe as dd

# Define known dtype issues explicitly
dtypes = {
    'a_offender': 'float64',
    'b_offender': 'float64',
    'completed_attempted2': 'object',
    'completed_attempted3': 'object',
    'female_offender': 'float64',
    'hour': 'object',
    'i_offender': 'float64',
    'male_offender': 'float64',
    'minor_offender': 'float64',
    'non_minor_offender': 'float64',
    'p_offender': 'float64',
    'property_description': 'object',
    'property_description2': 'object',
    'property_description3': 'object',
    'w_offender': 'float64'
}

# Read with Dask using the above dtype hints
df_dask = dd.read_csv(file_path, dtype=dtypes)

# Check number of rows and columns
print("Number of rows (approx):", df_dask.shape[0].compute())
print("Number of columns:", df_dask.shape[1])


/usr/local/lib/python3.11/dist-packages/dask/dataframe/io/csv.py:199: DtypeWarning: Columns (52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = reader(bio, **kwargs)


Number of rows (approx): 13683930
Number of columns: 53


In [10]:
import re

# Clean column names: remove special characters and extra whitespaces
cleaned_columns = [re.sub(r'\W+', '_', col).strip() for col in df_dask.columns]
df_dask.columns = cleaned_columns

# Show cleaned column names
df_dask.columns

Index(['state', 'ID', 'ORI', 'incident_number', 'date_HRF', 'date_SIF', 'hour',
       'total_offense', 'total_victim', 'total_offender', 'violence_offense',
       'theft_offense', 'drug_offense', 'sex_offense',
       'kidnapping_trafficking', 'other_offense', 'gun_involvement',
       'drug_involvement', 'property_value', 'stolen_motor', 'male_victim',
       'female_victim', 'unknown_sex_victim', 'w_victim', 'b_victim',
       'i_victim', 'a_victim', 'p_victim', 'unknown_race_victim',
       'minor_victim', 'non_minor_victim', 'unknown_age_victim',
       'offender_wi_family', 'offender_outside_family', 'offender_not_known',
       'male_offender', 'female_offender', 'unknown_sex_offender',
       'w_offender', 'b_offender', 'i_offender', 'a_offender', 'p_offender',
       'unknown_race_offender', 'minor_offender', 'non_minor_offender',
       'unknown_age_offender', 'completed_attempted2', 'completed_attempted3',
       'property_description', 'property_description2',
       'prop

In [11]:
# Store column names to use later in YAML schema
columns_list = list(df_dask.columns)
columns_list


['state',
 'ID',
 'ORI',
 'incident_number',
 'date_HRF',
 'date_SIF',
 'hour',
 'total_offense',
 'total_victim',
 'total_offender',
 'violence_offense',
 'theft_offense',
 'drug_offense',
 'sex_offense',
 'kidnapping_trafficking',
 'other_offense',
 'gun_involvement',
 'drug_involvement',
 'property_value',
 'stolen_motor',
 'male_victim',
 'female_victim',
 'unknown_sex_victim',
 'w_victim',
 'b_victim',
 'i_victim',
 'a_victim',
 'p_victim',
 'unknown_race_victim',
 'minor_victim',
 'non_minor_victim',
 'unknown_age_victim',
 'offender_wi_family',
 'offender_outside_family',
 'offender_not_known',
 'male_offender',
 'female_offender',
 'unknown_sex_offender',
 'w_offender',
 'b_offender',
 'i_offender',
 'a_offender',
 'p_offender',
 'unknown_race_offender',
 'minor_offender',
 'non_minor_offender',
 'unknown_age_offender',
 'completed_attempted2',
 'completed_attempted3',
 'property_description',
 'property_description2',
 'property_description3',
 'race_offender2']

# Validation

In [12]:
import pandas as pd
import dask.dataframe as dd
import logging
import os
import subprocess
import yaml
import datetime
import gc
import re
import csv
import gzip

In [13]:
%%writefile utility.py
import logging
import yaml
import re

def replacer(col, pattern):
    return re.sub(pattern, '', col)

def read_config_file(filepath):
    with open(filepath, 'r') as stream:
        try:
            return yaml.load(stream, Loader=yaml.Loader)
        except yaml.YAMLError as exc:
            logging.error(exc)

def col_header_val(df, table_config):
    df.columns = df.columns.str.lower()
    df.columns = df.columns.str.replace('[^\w]', '_', regex=True)
    df.columns = list(map(lambda x: x.strip('_'), list(df.columns)))
    df.columns = list(map(lambda x: replacer(x, '_'), list(df.columns)))

    expected_col = list(map(lambda x: x.lower(), table_config['columns']))
    expected_col.sort()
    df.columns = list(map(lambda x: x.lower(), list(df.columns)))
    df = df.reindex(sorted(df.columns), axis=1)

    if len(df.columns) == len(expected_col) and list(expected_col) == list(df.columns):
        print("Column name and length validation passed")
        return 1
    else:
        print("Column name and length validation failed")
        mismatched_columns_file = list(set(df.columns).difference(expected_col))
        print("File columns not in YAML:", mismatched_columns_file)
        missing_yaml_columns = list(set(expected_col).difference(df.columns))
        print("YAML columns not in file:", missing_yaml_columns)
        logging.info(f'df columns: {df.columns}')
        logging.info(f'expected columns: {expected_col}')
        return 0


Overwriting utility.py


In [14]:
%%writefile store.yaml
file_type: csv
dataset_name: file
file_name: Individual_Incident_2022
table_name: incident_data
inbound_delimiter: ","
outbound_delimiter: "|"
skip_leading_rows: 1
columns:
  - state
  - ID
  - ORI
  - incident_number
  - date_HRF
  - date_SIF
  - hour
  - total_offense
  - total_victim
  - total_offender
  - violence_offense
  - theft_offense
  - drug_offense
  - sex_offense
  - kidnapping_trafficking
  - other_offense
  - gun_involvement
  - drug_involvement
  - property_value
  - stolen_motor
  - male_victim
  - female_victim
  - unknown_sex_victim
  - w_victim
  - b_victim
  - i_victim
  - a_victim
  - p_victim
  - unknown_race_victim
  - minor_victim
  - non_minor_victim
  - unknown_age_victim
  - offender_wi_family
  - offender_outside_family
  - offender_not_known
  - male_offender
  - female_offender
  - unknown_sex_offender
  - w_offender
  - b_offender
  - i_offender
  - a_offender
  - p_offender
  - unknown_race_offender
  - minor_offender
  - non_minor_offender
  - unknown_age_offender
  - completed_attempted2
  - completed_attempted3
  - property_description
  - property_description2
  - property_description3
  - race_offender2

Overwriting store.yaml


In [15]:
import importlib
import utility
importlib.reload(utility)

<module 'utility' from '/content/utility.py'>

In [16]:
config_data = utility.read_config_file("store.yaml")

In [17]:
file_type = config_data['file_type']
source_file = "/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022." + file_type

In [18]:
dtype_map = {
    'a_offender': 'float64',
    'b_offender': 'float64',
    'completed_attempted2': 'object',
    'completed_attempted3': 'object',
    'female_offender': 'float64',
    'hour': 'object',
    'i_offender': 'float64',
    'male_offender': 'float64',
    'minor_offender': 'float64',
    'non_minor_offender': 'float64',
    'p_offender': 'float64',
    'property_description': 'object',
    'property_description2': 'object',
    'property_description3': 'object',
    'w_offender': 'float64'
}

df_sample = dd.read_csv(
    source_file,
    delimiter=config_data['inbound_delimiter'],
    dtype=dtype_map
)
df_sample.head()


,state,ID,ORI,incident_number,date_HRF,date_SIF,hour,total_offense,total_victim,total_offender,...,unknown_race_offender,minor_offender,non_minor_offender,unknown_age_offender,completed_attempted2,completed_attempted3,property_description,property_description2,property_description3,race_offender2
0,AK-Alaska,AK0010200_0G1A0BRVSCTD,AK0010200,0G1A0BRVSCTD,20220625,25jun2022,<NA>,NaN,NaN,NaN,...,NaN,1.0,0.0,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,
1,AK-Alaska,AK0010200_0G1G0BRVSCTD,AK0010200,0G1G0BRVSCTD,20220713,13jul2022,<NA>,NaN,NaN,NaN,...,NaN,1.0,0.0,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,
2,AK-Alaska,AK0010200_0G1K0BRVSCTD,AK0010200,0G1K0BRVSCTD,20220702,02jul2022,<NA>,NaN,NaN,NaN,...,NaN,1.0,0.0,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,
3,AK-Alaska,AK0010200_0G1N0BRVSCTD,AK0010200,0G1N0BRVSCTD,20220711,11jul2022,<NA>,NaN,NaN,NaN,...,NaN,1.0,0.0,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,
4,AK-Alaska,AK0010200_0G9A0BRVSCTD,AK0010200,0G9A0BRVSCTD,20220526,26apr2022,<NA>,NaN,NaN,NaN,...,NaN,1.0,0.0,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,


In [19]:
chunk_iter = pd.read_csv(source_file, delimiter=config_data['inbound_delimiter'], chunksize=100_000)
first_chunk = next(chunk_iter)
first_chunk.head()


,state,ID,ORI,incident_number,date_HRF,date_SIF,hour,total_offense,total_victim,total_offender,...,unknown_race_offender,minor_offender,non_minor_offender,unknown_age_offender,completed_attempted2,completed_attempted3,property_description,property_description2,property_description3,race_offender2
0,AK-Alaska,AK0010200_0G1A0BRVSCTD,AK0010200,0G1A0BRVSCTD,20220625,25jun2022,NaN,NaN,NaN,NaN,...,NaN,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,
1,AK-Alaska,AK0010200_0G1G0BRVSCTD,AK0010200,0G1G0BRVSCTD,20220713,13jul2022,NaN,NaN,NaN,NaN,...,NaN,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,
2,AK-Alaska,AK0010200_0G1K0BRVSCTD,AK0010200,0G1K0BRVSCTD,20220702,02jul2022,NaN,NaN,NaN,NaN,...,NaN,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,
3,AK-Alaska,AK0010200_0G1N0BRVSCTD,AK0010200,0G1N0BRVSCTD,20220711,11jul2022,NaN,NaN,NaN,NaN,...,NaN,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,
4,AK-Alaska,AK0010200_0G9A0BRVSCTD,AK0010200,0G9A0BRVSCTD,20220526,26apr2022,NaN,NaN,NaN,NaN,...,NaN,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,


In [20]:
# Validate header
import utility as util

if util.col_header_val(first_chunk, config_data) == 0:
    print("Validation failed")
else:
    print("Column validation passed")


Column name and length validation failed
File columns not in YAML: ['aoffender', 'otheroffense', 'nonminorvictim', 'druginvolvement', 'unknownraceoffender', 'theftoffense', 'propertyvalue', 'ioffender', 'violenceoffense', 'ivictim', 'minorvictim', 'sexoffense', 'avictim', 'woffender', 'boffender', 'unknownsexvictim', 'incidentnumber', 'guninvolvement', 'raceoffender2', 'propertydescription', 'datehrf', 'offenderwifamily', 'totalvictim', 'bvictim', 'wvictim', 'minoroffender', 'offenderoutsidefamily', 'totaloffense', 'datesif', 'propertydescription2', 'pvictim', 'completedattempted3', 'completedattempted2', 'totaloffender', 'drugoffense', 'kidnappingtrafficking', 'maleoffender', 'nonminoroffender', 'stolenmotor', 'femalevictim', 'unknownagevictim', 'poffender', 'unknownsexoffender', 'femaleoffender', 'propertydescription3', 'offendernotknown', 'unknownageoffender', 'unknownracevictim', 'malevictim']
YAML columns not in file: ['a_offender', 'property_description3', 'female_victim', 'i_vic

In [21]:
print("Columns in file are:\n", list(first_chunk.columns))
print("Columns in YAML are:\n", config_data['columns'])


Columns in file are:
 ['state', 'id', 'ori', 'incidentnumber', 'datehrf', 'datesif', 'hour', 'totaloffense', 'totalvictim', 'totaloffender', 'violenceoffense', 'theftoffense', 'drugoffense', 'sexoffense', 'kidnappingtrafficking', 'otheroffense', 'guninvolvement', 'druginvolvement', 'propertyvalue', 'stolenmotor', 'malevictim', 'femalevictim', 'unknownsexvictim', 'wvictim', 'bvictim', 'ivictim', 'avictim', 'pvictim', 'unknownracevictim', 'minorvictim', 'nonminorvictim', 'unknownagevictim', 'offenderwifamily', 'offenderoutsidefamily', 'offendernotknown', 'maleoffender', 'femaleoffender', 'unknownsexoffender', 'woffender', 'boffender', 'ioffender', 'aoffender', 'poffender', 'unknownraceoffender', 'minoroffender', 'nonminoroffender', 'unknownageoffender', 'completedattempted2', 'completedattempted3', 'propertydescription', 'propertydescription2', 'propertydescription3', 'raceoffender2']
Columns in YAML are:
 ['state', 'ID', 'ORI', 'incident_number', 'date_HRF', 'date_SIF', 'hour', 'total_o

In [22]:
import os

# Path to the conflicting file
conflicting_file = "/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022.csv.gz"

# Delete the file if it exists
if os.path.isfile(conflicting_file):
    os.remove(conflicting_file)
    print("Old file removed.")
else:
    print("No conflicting file found.")

No conflicting file found.


In [24]:
from dask import dataframe as dd
import os
import csv

output_dir = "/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022.csv.gz"
os.makedirs(output_dir, exist_ok=True)

# Read full data with Dask

df_dask = dd.read_csv(source_file, delimiter=config_data['inbound_delimiter'], dtype=dtype_map)

# Write gzip-compressed, pipe-delimited multi-part CSV

df_dask.to_csv(
    output_dir,
    sep='|',
    header=True,
    index=False,
    quoting=csv.QUOTE_ALL,
    compression='gzip',
    quotechar='"',
    doublequote=True
)


/usr/local/lib/python3.11/dist-packages/dask/dataframe/io/csv.py:199: DtypeWarning: Columns (52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = reader(bio, **kwargs)


['/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022.csv.gz/00.part',
 '/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022.csv.gz/01.part',
 '/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022.csv.gz/02.part',
 '/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022.csv.gz/03.part',
 '/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022.csv.gz/04.part',
 '/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022.csv.gz/05.part',
 '/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022.csv.gz/06.part',
 '/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022.csv.gz/07.part',
 '/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022.csv.gz/08.part',
 '/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022.csv.gz/09.part',
 '/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022.csv.gz/10.part',
 '/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022.csv.gz/11.part',
 '/content/drive

In [30]:
print("Generated part files:")
entries = os.listdir(output_dir)
for entry in entries:
    print(entry)


Generated part files:
32.part
33.part
08.part
10.part
23.part
29.part
18.part
15.part
19.part
02.part
35.part
20.part
05.part
12.part
13.part
07.part
00.part
22.part
26.part
01.part
11.part
24.part
28.part
27.part
21.part
17.part
06.part
31.part
25.part
14.part
36.part
34.part
30.part
04.part
03.part
09.part
37.part
16.part


In [26]:
#size of the gz format folder
os.path.getsize('/content/drive/MyDrive/Data Ingestion/Individual_Incident_2022.csv.gz')

4096